Projeto de Machine Learning - Detecção de Fraude em Seguros Automotivos

Este projeto tem como objetivo aplicar técnicas básicas de ciência de dados e aprendizado de máquina utilizando um dataset de sinistros de seguros automotivos.

O conjunto de dados contém informações sobre clientes, veículos, características do seguro e a indicação se foi identificado fraude ou não.

As etapas do projeto são:

Análise inicial dos dados – leitura, inspeção e limpeza do dataset.
Tratamento e preparação – transformação de variáveis e criação de um dataset limpo para modelagem.
Análise exploratória – construção de gráficos e estatísticas descritivas.
Modelagem preditiva – uso de Regressão Logística para prever a probabilidade de fraude.
Avaliação – análise de métricas de desempenho do modelo.

Este notebook foi desenvolvido como parte de um estudo prático para consolidar conhecimentos em ciência de dados aplicada a problemas reais.

In [ ]:
import pandas as pd

# Carregar dataset
df = pd.read_csv("data/carclaims.csv")

# Visualizar primeiras linhas
print(df.head())

# Info geral do dataset
print(df.info())

# Estatísticas descritivas
print(df.describe())

## Análise Inicial do Dataset

Nesta etapa realizei a **leitura e inspeção inicial** do dataset de sinistros de seguros automotivos.

- `df.head()` nos permite visualizar as primeiras linhas do dataset, dando uma ideia dos tipos de dados e do conteúdo das colunas.
- `df.info()` mostra informações gerais sobre o dataset, como número de linhas, tipos de dados e contagem de valores não nulos.
- `df.describe()` apresenta **estatísticas descritivas** das variáveis numéricas, como média, mínimo, máximo e quartis.

Essas informações ajudam a **identificar problemas potenciais** nos dados, como valores ausentes, tipos de dados incorretos ou colunas que precisam de tratamento antes da modelagem.


In [ ]:
import pandas as pd

# 1. Carregar dataset
df = pd.read_csv("data/carclaims.csv")


# 2. Visão geral
print("Shape do dataset:", df.shape)
print("\nValores nulos por coluna:\n", df.isnull().sum())
print("\nDuplicados:", df.duplicated().sum())

# 3. Remover duplicados
df = df.drop_duplicates()

# 4. Tratar valores nulos
# Categorias → "Desconhecido", Numéricos → mediana
categorical_cols = df.select_dtypes(include=["object"]).columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in categorical_cols:
    df[col] = df[col].fillna("Desconhecido")

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# 5. Converter colunas categóricas em category
for col in categorical_cols:
    df[col] = df[col].astype("category")

# 6. Salvar dataset limpo
df.to_csv("data/carclaims_clean.csv", index=False)
print("\nDataset limpo salvo em 'data/carclaims_clean.csv'")

# 7. Conferir resultado final
print("\nShape final:", df.shape)
print("\nColunas finais:", df.columns.tolist())

## Limpeza e Preparação dos Dados

Nesta etapa realizei o **tratamento do dataset** para deixá-lo pronto para análise e modelagem:

1. **Visão geral**: verificamos o tamanho do dataset (`shape`), valores nulos e duplicados.
2. **Remoção de duplicados**: registros repetidos foram excluídos para evitar vieses na modelagem.
3. **Tratamento de valores nulos**:
   - Para colunas categóricas, valores nulos foram preenchidos com `"Desconhecido"`.
   - Para colunas numéricas, valores nulos foram preenchidos com a **mediana** da coluna.
4. **Conversão de tipos**: colunas categóricas foram convertidas para o tipo `category` do Pandas, facilitando a codificação para modelagem.
5. **Salvar dataset limpo**: o arquivo final foi salvo como `carclaims_clean.csv` na pasta `data`.

Essas etapas garantem que o dataset esteja **consistente, sem valores faltantes ou duplicados**, pronto para a análise exploratória e modelagem de machine learning.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Carregar dataset limpo
df = pd.read_csv("data/carclaims_clean.csv")

# 2. Separar variáveis preditoras (X) e alvo (y)
X = df.drop("FraudFound", axis=1)
y = df["FraudFound"]

# 3. Dividir em treino e teste (70% treino, 30% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 4. Criar e treinar o modelo
model = RandomForestClassifier(random_state=42, class_weight="balanced")
model.fit(X_train, y_train)

# 5. Fazer previsões
y_pred = model.predict(X_test)

# 6. Avaliar o modelo
print("\n--- Avaliação do Modelo ---")
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))

print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred))

# 7. Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Não Fraude", "Fraude"], yticklabels=["Não Fraude", "Fraude"])
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão")
plt.show()


## Modelagem Preditiva com Random Forest

Nesta etapa construí um **modelo de Machine Learning** para prever fraudes em sinistros de seguros automotivos.

1. **Separação das variáveis**:
   - `X`: variáveis preditoras (todos os dados exceto a coluna alvo `FraudFound`).
   - `y`: variável alvo (`FraudFound`), indicando se houve fraude.

2. **Divisão em treino e teste**:
   - 70% dos dados para treino e 30% para teste.
   - Utilizamos `stratify=y` para manter a proporção da variável alvo em ambos os conjuntos.

3. **Criação do modelo**:
   - Usamos **Random Forest** com `class_weight="balanced"` para lidar com a desproporção entre fraudes e não-fraudes.
   - O modelo foi treinado com os dados de treino (`fit`).

4. **Avaliação do modelo**:
   - Métricas principais: **Acurácia, Precisão, Recall e F1-Score**.
   - `classification_report` fornece uma visão detalhada por classe.
   - **Matriz de Confusão** mostra o desempenho do modelo na classificação de fraudes e não-fraudes.

Esses passos permitem analisar se o modelo consegue identificar fraudes de forma eficaz e se pode ser utilizado em um cenário real de detecção de sinistros fraudulentos.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Carregar dataset limpo
df = pd.read_csv("data/carclaims_clean.csv")

# Separar features e target
X = df.drop("FraudFound", axis=1)
y = df["FraudFound"].map({"No": 0, "Yes": 1})

# Transformar variáveis categóricas
X = pd.get_dummies(X, drop_first=True)

# Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Escalar dados
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Treinar modelo
model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(X_train, y_train)

# Previsões
y_pred = model.predict(X_test)

# Avaliar
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nMatriz de Confusão:\n", confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred, zero_division=0))

# -----------------------
# Visualização de métricas
# -----------------------

# Matriz de Confusão como Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Fraud","Fraud"], yticklabels=["No Fraud","Fraud"])
plt.ylabel("True")
plt.xlabel("Predicted")
plt.title("Matriz de Confusão")
plt.show()

# Distribuição de previsões
plt.figure(figsize=(5,3))
sns.countplot(x=y_pred)
plt.xticks([0,1], ["No Fraud","Fraud"])
plt.title("Distribuição das Previsões")
plt.show()

# Comparar fraude real x prevista
plt.figure(figsize=(6,4))
df_compare = pd.DataFrame({"Real": y_test, "Predito": y_pred})
sns.countplot(x="Real", hue="Predito", data=df_compare)
plt.xticks([0,1], ["No Fraud","Fraud"])
plt.title("Fraude Real vs Previsão")
plt.show()


## Modelo de Regressão Logística e Visualização de Métricas

Nesta etapa construi um **modelo de Regressão Logística** para prever fraudes, com foco na análise detalhada das previsões.

1. **Preparação dos dados**:
   - Variáveis categóricas transformadas em dummies (`get_dummies`).
   - Escalonamento dos dados numéricos (`StandardScaler`) para melhorar a performance da regressão logística.

2. **Treinamento do modelo**:
   - Regressão Logística com `class_weight="balanced"` para lidar com o desbalanceamento entre fraudes e não-fraudes.
   - Divisão dos dados em treino (70%) e teste (30%) mantendo a proporção da variável alvo.

3. **Avaliação do modelo**:
   - Métricas: **Acurácia, Matriz de Confusão e Relatório de Classificação**.
   - `classification_report` detalha precisão, recall e F1-score para cada classe.

4. **Visualizações**:
   - **Matriz de Confusão** em heatmap, mostrando acertos e erros por classe.
   - **Distribuição das previsões** para verificar o balanceamento do modelo.
   - **Comparação entre fraude real x prevista**, destacando onde o modelo acerta e onde erra.

Essas análises permitem **entender melhor o desempenho do modelo**, identificar vieses e interpretar como ele poderia ser usado para detectar fraudes em um cenário real.
